# Reproduce the Expanded32 Main Pipeline

This notebook is the public reproduction entry point for the expanded32 **PortWatch--GDELT--WITS** pipeline.

It documents the main workflow for a reliability-aware ML-NLP-Network benchmark: PortWatch provides operational disruption labels and baselines, GDELT provides external event signals, and WITS provides network exposure and placebo-audit structure.

## Research Design

The benchmark asks whether external event signals improve next-week abnormal container port activity prediction beyond operational baselines, and whether WITS-weighted exposure improves or audits those event signals relative to unweighted and placebo controls.

The public main line is:

- **Operational only**: PortWatch history and vulnerability features.
- **Operational + GDELT**: external event signal added to operational features.
- **Operational + GDELT + WITS additive**: true network exposure added as an additive feature block.
- **True WITS gated / placebo gated**: network-audited event-conversion checks against equal, random, and shuffled WITS placebos.
- **Guarded / high-confidence deployment policies**: validation-safe policies for top-k alerting and APRS-style reliability evaluation.

Evaluation uses temporal validation. PR-AUC is central because abnormal port activity is rare.

## Prerequisites

The public repository does not commit large raw/interim data files. To run the expanded32 workflow end to end, the following local caches should exist under `data/interim/`:

- `gkg_partner_event_features_2021-01-01_2025-12-31_expanded32.csv`
- `gkg_partner_me_strict_event_features_2021-01-01_2025-12-31_expanded32.csv`
- `panel32_total_dependency_weights_2023.csv`

The GDELT files are generated from partition-filtered BigQuery queries with dry-run cost checks. The WITS weights can be regenerated with the WITS fetch script when API access is available.

A sanitized table-level result snapshot is committed under `results/` for readers who want to inspect the current evidence without local data caches.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

In [ ]:
required_cache_files = [
    PROJECT_ROOT / "data" / "interim" / "gkg_partner_event_features_2021-01-01_2025-12-31_expanded32.csv",
    PROJECT_ROOT / "data" / "interim" / "gkg_partner_me_strict_event_features_2021-01-01_2025-12-31_expanded32.csv",
    PROJECT_ROOT / "data" / "interim" / "panel32_total_dependency_weights_2023.csv",
]

missing = [path for path in required_cache_files if not path.exists()]
if missing:
    print("Missing local caches. Full reproduction requires:")
    for path in missing:
        print(" -", path)
    print("
You can still inspect the committed public result snapshot under results/.")
else:
    for path in required_cache_files:
        size_mb = path.stat().st_size / 1_000_000
        print(f"Found {path.name}: {size_mb:.1f} MB")

## Step 1: Build the Expanded32 Benchmark Dataset

This step constructs the model-ready country-week panel. It combines PortWatch operational activity, cached GDELT event features, and WITS dependency weights, then writes the expanded32 processed dataset.

In [ ]:
def run_script(script_name):
    command = [sys.executable, str(PROJECT_ROOT / "scripts" / script_name)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)


if not missing:
    run_script("build_expanded32_panel_benchmark_dataset.py")
else:
    print("Skipping dataset build because required local caches are missing.")

In [ ]:
dataset_path = PROJECT_ROOT / "data" / "processed" / "multicountry32_container_event_network_benchmark.csv"

if dataset_path.exists():
    df = pd.read_csv(dataset_path, parse_dates=["week"])
    summary = {
        "rows": len(df),
        "countries": df["ISO3"].nunique(),
        "positive_labels": int(df["abnormal_next_week_container"].sum()),
        "positive_rate": round(df["abnormal_next_week_container"].mean(), 4),
        "min_week": df["week"].min().date(),
        "max_week": df["week"].max().date(),
    }
    display(summary)
else:
    public_summary = PROJECT_ROOT / "results" / "tables" / "data_task_summary.csv"
    display(pd.read_csv(public_summary))

## Step 2: Run the Expanded32 Benchmark Models

This step evaluates the public expanded32 benchmark ladder. Thresholds are selected on validation periods only and then applied to later test periods.

In [ ]:
if dataset_path.exists():
    run_script("run_panel32_benchmark_models.py")
else:
    print("Skipping model run because the expanded32 processed dataset is missing.")

In [ ]:
summary_path = PROJECT_ROOT / "reports" / "tables" / "panel32_benchmark_summary.csv"
public_ladder_path = PROJECT_ROOT / "results" / "tables" / "main_model_ladder.csv"

if summary_path.exists():
    model_summary = pd.read_csv(summary_path)
    display(
        model_summary.sort_values(["model", "mean_pr_auc"], ascending=[True, False])[
            [
                "feature_group",
                "model",
                "mean_pr_auc",
                "std_pr_auc",
                "mean_roc_auc",
                "mean_f1",
                "mean_precision",
                "mean_recall",
                "total_tp",
                "total_fp",
                "total_fn",
            ]
        ].head(20)
    )
else:
    display(pd.read_csv(public_ladder_path))

## Step 3: Run Network Audit and Deployment Checks

The core paper-facing checks compare true WITS exposure against equal/random/shuffled placebos and evaluate guarded or high-confidence deployment policies. These scripts require the local generated prediction tables from the preceding workflow.

In [ ]:
if dataset_path.exists():
    for script in [
        "run_panel32_network_gated_conversion_main.py",
        "run_panel32_gdelt_conversion_propensity_benchmark.py",
        "run_panel32_country_shared_alert_allocation.py",
        "run_panel32_high_confidence_alert_policy.py",
    ]:
        run_script(script)
else:
    print("Skipping audit/deployment scripts because the expanded32 dataset is missing.")

In [ ]:
for public_table in [
    PROJECT_ROOT / "results" / "tables" / "network_audit.csv",
    PROJECT_ROOT / "results" / "tables" / "high_confidence_alert_policy.csv",
]:
    print(public_table.name)
    display(pd.read_csv(public_table))

## Step 4: Final Reproducibility Check

The tests are intentionally lightweight and offline. They use a committed tiny fixture dataset, so they can run after a fresh clone without private data caches.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
    cwd=PROJECT_ROOT,
    check=True,
)

## Expected Outputs

Main local generated artifacts:

- `data/processed/multicountry32_container_event_network_benchmark.csv`
- `reports/panel32_benchmark_dataset_summary.md`
- `reports/panel32_benchmark_results.md`
- `reports/tables/panel32_benchmark_summary.csv`
- `reports/tables/main_paper_consolidated_model_ladder.csv`
- `reports/tables/main_paper_network_audit_checks.csv`
- `reports/tables/main_paper_aprs_scores.csv`

Committed public snapshot:

- `results/README.md`
- `results/tables/main_model_ladder.csv`
- `results/tables/network_audit.csv`
- `results/tables/high_confidence_alert_policy.csv`

Interpret results cautiously: WITS is an exposure-mapping and audit layer, not causal propagation evidence, and true WITS does not consistently dominate placebo networks.